# Lecture 4 · Exercises

Code for [exercises.md](exercises.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

DATA = Path("../../coding_tasks/Data/Out_Data")
rng = np.random.default_rng(1234)

---

## Exercise 4.1 · Threats to internal validity

No code, see [exercises.md](exercises.md)

---

## Exercise 4.2 · Schooling and earnings

### Data

In [ ]:
df = pd.read_csv(DATA / "lecture4_schooling_earnings.csv")
print(df.shape)
df.head()

In [ ]:
df.describe().round(2)

Schooling and motivation move together almost perfectly. That is what parts 4 and 5 are about.

In [ ]:
df.corr().round(3)

### 2 · Scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df["years_schooling"], df["earnings"], s=8, color="#4C78A8", alpha=0.7)
ax.set_xlabel("years of schooling")
ax.set_ylabel("earnings (CHF)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Tight, positive, linear. No curvature, so no functional form problem.

### 3 · Guess the slope

Read two points off the graph and divide. The group means around 15 and 25 years show the eyeball estimate is close.

In [ ]:
for years in [15, 25]:
    sub = df[df["years_schooling"].between(years - 0.5, years + 0.5)]
    print(f"{years} years: n = {len(sub):>3}, mean earnings = {sub['earnings'].mean():8.1f}")

guess = (10_000 - 6_500) / 10
print(f"\nguess from the graph: {guess:.0f} CHF per year")

### 4 · Short regression

In [ ]:
short = smf.ols("earnings ~ years_schooling", data=df).fit()
print(short.summary().tables[1])
print(f"R2 = {short.rsquared:.4f}, residual SE = {np.sqrt(short.scale):.2f}")

### 5 · Adding motivation

In [ ]:
long = smf.ols("earnings ~ years_schooling + motivation", data=df).fit()
print(long.summary().tables[1])
print(f"R2 = {long.rsquared:.6f}, residual SE = {np.sqrt(long.scale):.4f}")

In [ ]:
pd.DataFrame(
    {
        "without motivation": short.params.reindex(["Intercept", "years_schooling", "motivation"]),
        "with motivation": long.params.reindex(["Intercept", "years_schooling", "motivation"]),
    }
).round(3)

The coefficient on schooling drops from 339 to 150. Motivation was hiding inside it.

Omitted variable bias says the gap is the effect of motivation on earnings times the slope of motivation on schooling.

In [ ]:
aux = smf.ols("motivation ~ years_schooling", data=df).fit()
gamma = long.params["motivation"]
delta = aux.params["years_schooling"]

print(f"gamma (motivation -> earnings)    = {gamma:.4f}")
print(f"delta (schooling -> motivation)   = {delta:.4f}")
print(f"gamma * delta                     = {gamma * delta:.4f}")
print(f"short - long                      = {short.params['years_schooling'] - long.params['years_schooling']:.4f}")

The residual SE of the long regression is 1.0 and its R2 is 1. The data were simulated from

    earnings = 1500 + 150 * years_schooling + 200 * motivation + N(0, 1)

which is why the coefficients land on round numbers.

---

## Beyond the exercise · simulating the classical threats

The slides show the threats as pictures. Simulating them is the quickest way to check the two rules that are easy to mix up in the exam:

- the **direction** of the bias from a confounder is the sign of Cor(Z, X) times Cor(Z, Y)
- measurement error in **X** biases towards zero, measurement error in **Y** only costs precision

### Direction of the bias from a confounder

Build data where the true effect of x on y is 2.0, with a confounder z whose two correlations can be set independently. Then regress y on x alone and compare.

In [ ]:
def confounded(sign_zx, sign_zy, n=5_000, beta=2.0):
    z = rng.normal(size=n)
    x = sign_zx * z + rng.normal(size=n)
    y = beta * x + sign_zy * 3 * z + rng.normal(size=n)
    d = pd.DataFrame({"x": x, "y": y, "z": z})
    return (
        smf.ols("y ~ x", data=d).fit().params["x"],
        smf.ols("y ~ x + z", data=d).fit().params["x"],
    )


rows = []
for sx, label_x in [(1, "positive"), (-1, "negative")]:
    for sy, label_y in [(1, "positive"), (-1, "negative")]:
        without_z, with_z = confounded(sx, sy)
        rows.append(
            {
                "Cor(Z, X)": label_x,
                "Cor(Z, Y)": label_y,
                "true beta": 2.0,
                "without Z": without_z,
                "with Z": with_z,
                "bias": without_z - 2.0,
                "direction": "upward" if without_z > 2.0 else "downward",
            }
        )

pd.DataFrame(rows).round(3)

Same signs push the estimate up, opposite signs push it down, exactly as in the table on slide 6. Controlling for z recovers 2.0 in all four cases.

### Measurement error in X and in Y

One draw cannot separate bias from luck, so repeat each design 500 times. True slope 3.0, x uniform on [0, 10].

In [ ]:
N, REPS, BETA = 1_000, 500, 3.0


def slope(x, y):
    xc = x - x.mean()
    return (xc * (y - y.mean())).sum() / (xc ** 2).sum()


estimates = {k: [] for k in ["clean", "X error, sd 3", "X error, sd 6", "Y error, sd 10", "Y error, sd 20"]}

for _ in range(REPS):
    x = rng.uniform(0, 10, N)
    y = 30 + BETA * x + rng.normal(0, 2, N)
    estimates["clean"].append(slope(x, y))
    estimates["X error, sd 3"].append(slope(x + rng.normal(0, 3, N), y))
    estimates["X error, sd 6"].append(slope(x + rng.normal(0, 6, N), y))
    estimates["Y error, sd 10"].append(slope(x, y + rng.normal(0, 10, N)))
    estimates["Y error, sd 20"].append(slope(x, y + rng.normal(0, 20, N)))

estimates = {k: np.array(v) for k, v in estimates.items()}

pd.DataFrame(
    {
        "mean estimate": {k: v.mean() for k, v in estimates.items()},
        "bias": {k: v.mean() - BETA for k, v in estimates.items()},
        "standard error": {k: v.std() for k, v in estimates.items()},
    }
).round(3)

The attenuation is not a vague tendency, it is a known factor: the share of the observed variance in x that is real.

    plim beta_hat = beta * var(x) / (var(x) + var(u))

In [ ]:
var_x = 10 ** 2 / 12  # variance of a uniform on [0, 10]
for sd in [3, 6]:
    predicted = BETA * var_x / (var_x + sd ** 2)
    print(f"sd {sd}: predicted {predicted:.3f}, simulated {estimates[f'X error, sd {sd}'].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(len(estimates), 1, figsize=(8, 8), sharex=True)
bins = np.arange(0.0, 3.6, 0.02)
for ax, (name, est) in zip(axes, estimates.items()):
    ax.hist(est, bins=bins, color="#4C78A8", edgecolor="white", linewidth=0.5)
    ax.axvline(BETA, color="black", linewidth=1.5)
    ax.set_title(name, loc="left", fontsize=10)
    ax.set_yticks([])
    ax.spines[["top", "right", "left"]].set_visible(False)
axes[-1].set_xlabel("estimated slope (black line: true value 3.0)")
plt.tight_layout()
plt.show()

- **Error in x** shifts the whole distribution towards zero. No sample is close to 3.0, and more data would not help.
- **Error in y** leaves the distribution centred on 3.0 and only widens it. That is variance, not bias, and a larger sample fixes it.
- This is the same distinction as in lecture 3: a biased design cannot be rescued by sample size, an imprecise one can.